In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import json
import os

In [ ]:
DATA_DIR = "G:\\My Drive\\CEE247C\\247C Project\\data"
SF_FILE = "Footprint_Inventory_Combined_1.json"

# Load the data
with open(os.path.join(DATA_DIR, SF_FILE), 'r') as f:
    sf_all_data = json.load(f)

# Extract the features array from the GeoJSON
sf_df = pd.json_normalize(sf_all_data['features'])

FileNotFoundError: [Errno 2] No such file or directory: 'G:\\My Drive\\CEE 247C\\247C Project\\data\\Footprint_Inventory_Combined_1.json'

In [ ]:
sf_df.shape

(146445, 46)

In [ ]:
sf_df.columns

Index(['id', 'type', 'properties.FootprintID', 'properties.CensusBlock',
       'properties.CensusTract', 'properties.FootprintHeight',
       'properties.FootprintArea', 'properties.EstimatedStories',
       'properties.Total_SqFt', 'properties.Parcel Number',
       'properties.Property Location', 'properties.Use Code',
       'properties.Use Definition', 'properties.Property Class Code',
       'properties.Property Class Code Definition', 'properties.FC_Hazus',
       'properties.Year Property Built', 'properties.Number of Bathrooms',
       'properties.Number of Bedrooms', 'properties.Number of Rooms',
       'properties.Number of Stories', 'properties.Number of Units',
       'properties.Zoning Code', 'properties.Construction Type',
       'properties.Lot Depth', 'properties.Lot Frontage',
       'properties.Property Area', 'properties.Percent of Ownership',
       'properties.Exemption Code', 'properties.Current Sales Date',
       'properties.Assessed Fixtures Value',
       'pr

In [ ]:
col_filtered_sf_df = sf_df[['id','properties.Number of Stories', 'properties.Number of Units', 'properties.Year Property Built', 'properties.Construction Type', 'properties.POINT_NumPoints', 'properties.Soft Story', 'geometry.coordinates']].copy()

In [ ]:
# get the centroid of the geometry coordinates
col_filtered_sf_df['GPS_location'] = col_filtered_sf_df['geometry.coordinates'].apply(lambda x: [np.mean([point[0] for point in x[0]]), np.mean([point[1] for point in x[0]])])

In [ ]:
from pyproj import Transformer

# Create transformer from EPSG:26910 (UTM) to EPSG:4326 (WGS84 lat/lon)
transformer = Transformer.from_crs("EPSG:26910", "EPSG:4326", always_xy=True)

# Convert coordinates
def convert_to_latlon(coords):
    x, y = coords[0], coords[1]
    lon, lat = transformer.transform(x, y)
    return [lat, lon]

col_filtered_sf_df['GPS_location'] = col_filtered_sf_df['GPS_location'].apply(convert_to_latlon)

In [ ]:
# # unique values in Number of Units
# col_filtered_sf_df['properties.Number of Units'].value_counts()

In [ ]:
# simplify the column names-'properties.Number of Stories', 'properties.Number of Units', 'properties.Year Property Built', 'properties.Construction Type', 'properties.POINT_NumPoints', 'properties.Soft Story', 'geometry.coordinates']].copy()
col_filtered_sf_df = col_filtered_sf_df.rename(columns={
    'properties.Number of Stories': 'num_stories',
    'properties.EstimatedStories': 'estimated_stories',
    'properties.Number of Units': 'num_units',
    'properties.Year Property Built': 'year_built',
    'properties.Construction Type': 'construction_type',
    'properties.POINT_NumPoints': 'point_count',
    'properties.Soft Story': 'soft_story',
    'geometry.coordinates': 'geometry_coordinates'
})

In [ ]:
col_filtered_sf_df.head()

,id,num_stories,estimated_stories,num_units,year_built,construction_type,properties.POINT_ID,point_count,soft_story,geometry_coordinates,GPS_location
0,0,2.0,2,1,1929.0,D,178529,1.0,0.0,"[[[[545264.2646245655, 4178959.9308897792], [5...","[2362112.0977571723, 2362119.2709371876]"
1,1,3.0,4,5,1911.0,D,100406,1.0,1.0,"[[[[550700.6946592117, 4179988.7282696543], [5...","[2365344.711464433, 2365341.7437550975]"
2,2,2.0,4,2,1900.0,D,186143,1.0,0.0,"[[[[551428.1966417282, 4178653.2636145465], [5...","[2365040.7301281374, 2365044.932835309]"
3,3,1.0,2,1,1956.0,D,34205,1.0,0.0,"[[[[543887.7548867267, 4181505.144690213], [54...","[2362696.44978847, 2362695.1098743007]"
4,4,1.0,2,1,1929.0,D,193422,1.0,0.0,"[[[[548483.0896318282, 4174088.2492104517], [5...","[2361285.66942114, 2361285.63788863]"


In [ ]:
# Create a new column for "num_units_updated"
def calculate_num_units(row):
    try:
        if isinstance(row['num_units'], list):
            return sum(row['num_units']) 
        elif pd.isna(row['num_units']) or pd.isna(row['point_count']):
            return np.nan
        else:
            return float(row['num_units']) * float(row['point_count'])
    except (ValueError, TypeError):
        return np.nan

# Apply the function to create the new column
col_filtered_sf_df['num_units_updated'] = col_filtered_sf_df.apply(calculate_num_units, axis=1)

In [ ]:
col_filtered_sf_df.head()

,id,num_stories,estimated_stories,num_units,year_built,construction_type,properties.POINT_ID,point_count,soft_story,geometry_coordinates,GPS_location,num_units_updated
0,0,2.0,2,1,1929.0,D,178529,1.0,0.0,"[[[[545264.2646245655, 4178959.9308897792], [5...","[2362112.0977571723, 2362119.2709371876]",1.0
1,1,3.0,4,5,1911.0,D,100406,1.0,1.0,"[[[[550700.6946592117, 4179988.7282696543], [5...","[2365344.711464433, 2365341.7437550975]",5.0
2,2,2.0,4,2,1900.0,D,186143,1.0,0.0,"[[[[551428.1966417282, 4178653.2636145465], [5...","[2365040.7301281374, 2365044.932835309]",2.0
3,3,1.0,2,1,1956.0,D,34205,1.0,0.0,"[[[[543887.7548867267, 4181505.144690213], [54...","[2362696.44978847, 2362695.1098743007]",1.0
4,4,1.0,2,1,1929.0,D,193422,1.0,0.0,"[[[[548483.0896318282, 4174088.2492104517], [5...","[2361285.66942114, 2361285.63788863]",1.0


In [ ]:
# unique values in num_units_updated
col_filtered_sf_df['num_units_updated'].value_counts()

num_units_updated
1.0      93658
2.0      18139
0.0      10988
3.0       6777
4.0       4324
         ...  
134.0        1
89.0         1
147.0        1
209.0        1
241.0        1
Name: count, Length: 230, dtype: int64

In [ ]:
# Convert year_built to numeric, handling lists and non-numeric values
def extract_year(value):
    try:
        if isinstance(value, list):
            # Take the first year if it's a list
            return float(value[0]) if value else np.nan
        elif pd.isna(value):
            return np.nan
        else:
            return float(value)
    except (ValueError, TypeError, IndexError):
        return np.nan

col_filtered_sf_df['year_built'] = col_filtered_sf_df['year_built'].apply(extract_year)

# Do the same for num_stories if needed
def extract_numeric(value):
    try:
        if isinstance(value, list):
            return float(value[0]) if value else np.nan
        elif pd.isna(value):
            return np.nan
        else:
            return float(value)
    except (ValueError, TypeError, IndexError):
        return np.nan

col_filtered_sf_df['num_stories'] = col_filtered_sf_df['num_stories'].apply(extract_numeric)


In [ ]:
# Filter the data based on pre-1978, 3 or more stories, and 5 or more units
filtered_ordinance_sf_df = col_filtered_sf_df[
    (col_filtered_sf_df['year_built'] < 1978) &
    (col_filtered_sf_df['num_stories'] >= 3) &
    (col_filtered_sf_df['num_units_updated'] >= 5)
]

In [ ]:
filtered_ordinance_sf_df.shape

(6609, 12)

In [ ]:
filtered_ordinance_sf_df_soft_story = filtered_ordinance_sf_df[filtered_ordinance_sf_df['soft_story'] == 1] 
filtered_ordinance_sf_df_soft_story.shape

(3345, 12)

In [ ]:
filtered_ordinance_sf_df_not_soft_story = filtered_ordinance_sf_df[filtered_ordinance_sf_df['soft_story'] == 0]
filtered_ordinance_sf_df_not_soft_story.shape

(3264, 12)

In [ ]:
# save all the dataframes in a folder "Processed_SF_Data_Dataframes" 
output_dir = os.path.join(DATA_DIR, "Processed_SF_Data_Dataframes")
os.makedirs(output_dir, exist_ok=True)
filtered_ordinance_sf_df.to_csv(os.path.join(output_dir, "filtered_ordinance_sf_df.csv"), index=False)
filtered_ordinance_sf_df_soft_story.to_csv(os.path.join(output_dir, "filtered_ordinance_sf_df_soft_story.csv"), index=False)
filtered_ordinance_sf_df_not_soft_story.to_csv(os.path.join(output_dir, "filtered_ordinance_sf_df_not_soft_story.csv"), index=False)
col_filtered_sf_df.to_csv(os.path.join(output_dir, "col_filtered_sf_df.csv"), index=False)

In [ ]:
#Load the filtered_ordinance_sf_not_soft_story dataframe
loaded_filtered_ordinance_sf_df_not_soft_story = pd.read_csv(os.path.join(output_dir, "filtered_ordinance_sf_df_not_soft_story.csv"))
loaded_filtered_ordinance_sf_df_not_soft_story.shape

(3264, 12)